In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary

def double_conv(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=0), nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, 3, padding=0), nn.ReLU(inplace=True)
    )

class UNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super().__init__()
        self.enc1 = double_conv(in_channels, 64)
        self.enc2 = double_conv(64, 128)
        self.enc3 = double_conv(128, 256)
        self.enc4 = double_conv(256, 512)
        self.enc5 = double_conv(512, 1024)
        self.pool = nn.MaxPool2d(2)

        self.up6 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.up7 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.up8 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.up9 = nn.ConvTranspose2d(128, 64, 2, stride=2)

        self.dec6 = double_conv(1024, 512)
        self.dec7 = double_conv(512, 256)
        self.dec8 = double_conv(256, 128)
        self.dec9 = double_conv(128, 64)

        self.final = nn.Conv2d(64, num_classes, 1)

    def crop_center(self, x, target_h, target_w):
        _, _, h, w = x.shape
        return x[:, :, (h - target_h)//2 : (h + target_h)//2,
                    (w - target_w)//2 : (w + target_w)//2]

    def forward(self, x):
        # Encoder
        conv1 = self.enc1(x)                       # 568x568
        conv2 = self.enc2(self.pool(conv1))        # 280x280
        conv3 = self.enc3(self.pool(conv2))        # 136x136
        conv4 = self.enc4(self.pool(conv3))        # 64x64
        conv5 = self.enc5(self.pool(conv4))        # 28x28
        
# decoder
        up6 = self.up6(conv5)                      # 56x56
        crop4 = self.crop_center(conv4, 56, 56)    # crop 4 from each side
        conv6 = self.dec6(torch.cat([crop4, up6], 1))

        up7 = self.up7(conv6)                      # 104x104
        crop3 = self.crop_center(conv3, 104, 104)  # crop 16 from each side
        conv7 = self.dec7(torch.cat([crop3, up7], 1))

        up8 = self.up8(conv7)                      # 200x200
        crop2 = self.crop_center(conv2, 200, 200)  # crop 40 from each side
        conv8 = self.dec8(torch.cat([crop2, up8], 1))

        up9 = self.up9(conv8)                      # 392x392
        crop1 = self.crop_center(conv1, 392, 392)  # crop 88 from each side
        conv9 = self.dec9(torch.cat([crop1, up9], 1))

        return F.softmax(self.final(conv9), dim=1) # 388x388, channels=num_classes

In [10]:
model = UNet(in_channels=1, num_classes=2)
summary(model)  

Layer (type:depth-idx)                   Param #
UNet                                     --
├─Sequential: 1-1                        --
│    └─Conv2d: 2-1                       640
│    └─ReLU: 2-2                         --
│    └─Conv2d: 2-3                       36,928
│    └─ReLU: 2-4                         --
├─Sequential: 1-2                        --
│    └─Conv2d: 2-5                       73,856
│    └─ReLU: 2-6                         --
│    └─Conv2d: 2-7                       147,584
│    └─ReLU: 2-8                         --
├─Sequential: 1-3                        --
│    └─Conv2d: 2-9                       295,168
│    └─ReLU: 2-10                        --
│    └─Conv2d: 2-11                      590,080
│    └─ReLU: 2-12                        --
├─Sequential: 1-4                        --
│    └─Conv2d: 2-13                      1,180,160
│    └─ReLU: 2-14                        --
│    └─Conv2d: 2-15                      2,359,808
│    └─ReLU: 2-16                